# Explore GB National Demand

NESO historic `ND` plus live Insights INDO (the series we can validate every settlement period).

Demand-only. DST days, weekday vs holiday, the 2020–23 level shift, and lag autocorrelation — the choices used later in training and notebook 08.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import energy_forecast.data.demand as _demand
importlib.reload(_demand)
from energy_forecast.data.demand import load_demand

pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
demand = load_demand(include_live=True)
demand.tail()

In [ ]:
summary = demand.groupby("source")["demand_mw"].agg(rows="count", mean="mean", min="min", max="max")
summary = summary.join(
    demand.groupby("source")["settlement_date"].agg(first_date="min", last_date="max")
)
summary

In [ ]:
daily = (
    demand.groupby(["source", "settlement_date"], as_index=False)["demand_mw"]
    .mean()
    .sort_values("settlement_date")
)
fig, ax = plt.subplots(figsize=(12, 5))
for source, group in daily.groupby("source"):
    ax.plot(group["settlement_date"], group["demand_mw"], label=f"{source} daily mean", alpha=0.35, linewidth=1)
    rolled = group["demand_mw"].rolling(7, min_periods=3).mean()
    ax.plot(group["settlement_date"], rolled, label=f"{source} 7-day mean", linewidth=2)
ax.set_title("GB daily mean National Demand (7-day smooth)")
ax.set_ylabel("MW")
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
profile = demand.groupby("settlement_period", as_index=False)["demand_mw"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(profile["settlement_period"], profile["demand_mw"])
ax.set_title("Average GB demand by settlement period")
ax.set_xlabel("Settlement period")
ax.set_ylabel("MW")
plt.show()

## Coverage and DST

A London settlement day is 48 half-hours, except clocks-forward (46) and clocks-back (50). Missing days matter for `demand_lag_336`.

In [ ]:
work = demand.dropna(subset=["demand_mw"]).copy()
work["settlement_date"] = pd.to_datetime(work["settlement_date"]).dt.normalize()
per_day = work.groupby("settlement_date").size().rename("n_periods")
print("periods per day")
print(per_day.value_counts().sort_index().to_string())
print("days not 48 periods", int((per_day != 48).sum()))
expected = pd.date_range(per_day.index.min(), per_day.index.max(), freq="D")
missing = expected.difference(per_day.index)
print("missing calendar days", len(missing))
if len(missing):
    shown = missing[:10].strftime("%Y-%m-%d").tolist()
    print(shown, "..." if len(missing) > 10 else "")

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.scatter(per_day.index, per_day.to_numpy(), s=6, alpha=0.5)
ax.axhline(48, color="0.4", linewidth=0.8)
ax.set_title("Settlement periods per day (46 / 50 = clock change)")
ax.set_ylabel("periods")
fig.autofmt_xdate()
plt.show()

## Weekday, weekend, holiday

Same clock, different level. The Tuesday-after-holiday miss in notebook 08 is "yesterday was this low shape, today is not."

In [ ]:
import holidays

from energy_forecast.regions import GB
from energy_forecast.settlement import LONDON_TZ

stamp = pd.to_datetime(work["timestamp"], utc=True).dt.tz_convert(LONDON_TZ)
work["year"] = stamp.dt.year
work["day_of_week"] = stamp.dt.dayofweek
years = sorted(work["year"].dropna().unique().astype(int).tolist())
calendar = holidays.country_holidays("GB", subdiv=GB.holiday_subdiv, years=years)
work["is_holiday"] = work["settlement_date"].dt.date.map(lambda value: value in calendar)
work["day_type"] = "weekday"
work.loc[work["day_of_week"] >= 5, "day_type"] = "weekend"
work.loc[work["is_holiday"], "day_type"] = "holiday"

print(work.groupby("day_type")["demand_mw"].agg(n="count", mean="mean").to_string())

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [("weekday", "tab:blue"), ("weekend", "tab:orange"), ("holiday", "tab:red")]:
    profile = work.loc[work["day_type"] == label].groupby("settlement_period")["demand_mw"].mean()
    ax.plot(profile.index, profile.to_numpy(), label=label, color=color, linewidth=2)
ax.set_title("Mean demand by settlement period")
ax.set_xlabel("Settlement period")
ax.set_ylabel("MW")
ax.legend()
plt.show()

## Level by year

2020-23 sit in a different band (COVID, then the recovery). Week and day models drop those years from the fit so a 2026 Tuesday is not pulled toward 2020.

In [ ]:
yearly = work.groupby("year")["demand_mw"].agg(n="count", mean="mean", min="min", max="max")
display(yearly)

daily_year = work.groupby(["year", "settlement_date"], as_index=False)["demand_mw"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
for year, group in daily_year.groupby("year"):
    ax.plot(group["settlement_date"].dt.dayofyear, group["demand_mw"], label=str(int(year)), alpha=0.7, linewidth=1)
ax.set_title("Daily mean demand by day-of-year")
ax.set_xlabel("Day of year")
ax.set_ylabel("MW")
ax.legend(ncol=4, fontsize=8)
plt.show()

## Why the model is mostly lags

Correlation of demand with itself at the lags the models use. Weather will lose to these on almost every day.

In [ ]:
series = work.sort_values("timestamp")["demand_mw"]
lags = [(1, "30 min"), (2, "1 hour"), (48, "1 day"), (336, "1 week")]
lag_table = pd.DataFrame(
    [{"lag": lag, "label": label, "correlation": float(series.corr(series.shift(lag)))} for lag, label in lags]
)
display(lag_table)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lag_table["label"], lag_table["correlation"], color="tab:blue")
ax.set_ylim(0, 1)
ax.set_ylabel("Correlation")
ax.set_title("Demand autocorrelation at model lags")
plt.show()